In [4]:
from collections import defaultdict
import pandas as pd
import json
import os
import re
import hashlib

In [5]:
def filter_dataframe(df):
    valid_qp = {"0", "Yes"} | set(map(str, range(1, 6)))
    qp_clean = df["Query Project"].astype(str).str.strip()
    df_filtered = df[qp_clean.isin(valid_qp)]
    return df_filtered.copy()

In [ ]:
# Folder path
folder = "../../results/data"

# Get list of all CSV files in the folder
files = [f for f in os.listdir(folder) if f.endswith(".csv")]

# Read and concat
df = pd.concat([pd.read_csv(os.path.join(folder, f)) for f in files], ignore_index=True)

print(df.shape)

(1056081, 11)


In [7]:
# Sort by version descending, so the latest version comes first
df_sorted = df.sort_values(by="version", ascending=False)

# Keep only one row per hash
df_unique = df_sorted.drop_duplicates(subset="hash", keep="first").reset_index(drop=True)


In [9]:
print(df_unique.shape)

(462030, 11)


In [10]:
df_unique.head()

,hash,project_id,version,license,method_name,file_location,repository_url,query_project,violation,source_project,source_project_version
0,ca823f7ade8864d7a6f911a3b1afeed7,344147451,1.756905e+12,Other,__iter__,./terratorch/samplers/single.py:21,https://github.com/IBM/terratorch/blob/e7eb482...,3,NaN,NaN,NaN
1,ea1f822ff9040aeaa5540719a0c30b64,344147451,1.756905e+12,Other,import_custom_modules,./terratorch/cli_tools.py:173,https://github.com/IBM/terratorch/blob/e7eb482...,3,NaN,NaN,NaN
2,eab1760c25814d020fc25e37144b3d16,344147451,1.756905e+12,Other,__iter__,./terratorch/samplers/single.py:34,https://github.com/IBM/terratorch/blob/e7eb482...,3,NaN,NaN,NaN
3,26adeb4519d2341006421b47d8786c9b,2129832116,1.756894e+12,Apache License 2.0,get_predicate_id,./src/unitxt/metrics.py:2890,https://github.com/IBM/unitxt/blob/fa5b604db4c...,2,NaN,NaN,NaN
4,a8aa31281dac29e85402060e8065ab48,2793013759,1.756850e+12,Apache License 2.0,write,./social-network/social-network-source/gen-cpp...,https://github.com/IBM/DeepRest/blob/9c368eeef...,2,NaN,NaN,NaN


In [ ]:
# df.to_csv("../results/data/combined_all_methods.csv", index=False)

In [11]:
df = df_unique.iloc[:, :7]

In [12]:
df.head()

,hash,project_id,version,license,method_name,file_location,repository_url
0,ca823f7ade8864d7a6f911a3b1afeed7,344147451,1.756905e+12,Other,__iter__,./terratorch/samplers/single.py:21,https://github.com/IBM/terratorch/blob/e7eb482...
1,ea1f822ff9040aeaa5540719a0c30b64,344147451,1.756905e+12,Other,import_custom_modules,./terratorch/cli_tools.py:173,https://github.com/IBM/terratorch/blob/e7eb482...
2,eab1760c25814d020fc25e37144b3d16,344147451,1.756905e+12,Other,__iter__,./terratorch/samplers/single.py:34,https://github.com/IBM/terratorch/blob/e7eb482...
3,26adeb4519d2341006421b47d8786c9b,2129832116,1.756894e+12,Apache License 2.0,get_predicate_id,./src/unitxt/metrics.py:2890,https://github.com/IBM/unitxt/blob/fa5b604db4c...
4,a8aa31281dac29e85402060e8065ab48,2793013759,1.756850e+12,Apache License 2.0,write,./social-network/social-network-source/gen-cpp...,https://github.com/IBM/DeepRest/blob/9c368eeef...


In [13]:
def detect_language(path: str) -> str:
    # Remove any trailing line number (e.g. file.py:123 -> file.py)
    clean_path = re.sub(r":\d+$", "", str(path))
    
    ext = os.path.splitext(clean_path)[1].lower()
    
    mapping = {
        ".c": "c", ".h": "c",
        ".cpp": "cpp", ".cc": "cpp", ".hpp": "cpp",
        ".cs": "cs", ".java": "java",
        ".js": "js", ".ts": "js",
        ".py": "py",
    }
    return mapping.get(ext, "other")

In [14]:
df["Language"] = df["file_location"].apply(detect_language)

In [15]:
df.head()

,hash,project_id,version,license,method_name,file_location,repository_url,Language
0,ca823f7ade8864d7a6f911a3b1afeed7,344147451,1.756905e+12,Other,__iter__,./terratorch/samplers/single.py:21,https://github.com/IBM/terratorch/blob/e7eb482...,py
1,ea1f822ff9040aeaa5540719a0c30b64,344147451,1.756905e+12,Other,import_custom_modules,./terratorch/cli_tools.py:173,https://github.com/IBM/terratorch/blob/e7eb482...,py
2,eab1760c25814d020fc25e37144b3d16,344147451,1.756905e+12,Other,__iter__,./terratorch/samplers/single.py:34,https://github.com/IBM/terratorch/blob/e7eb482...,py
3,26adeb4519d2341006421b47d8786c9b,2129832116,1.756894e+12,Apache License 2.0,get_predicate_id,./src/unitxt/metrics.py:2890,https://github.com/IBM/unitxt/blob/fa5b604db4c...,py
4,a8aa31281dac29e85402060e8065ab48,2793013759,1.756850e+12,Apache License 2.0,write,./social-network/social-network-source/gen-cpp...,https://github.com/IBM/DeepRest/blob/9c368eeef...,cpp


In [16]:
def filter_trivial_functions_by_name(df, file_col="file_location"):
    if df.empty:
        return df.copy()

    if file_col not in df.columns:
        raise KeyError(f"Column '{file_col}' not found in DataFrame")

    df = df.copy()

    # --- Filter by Method Name format ---
    df = df[df['method_name'].str.match(r'^[A-Za-z_][A-Za-z0-9_]{2,}$', na=False)]

    # --- Filter by File Location format ---
    df = df[df[file_col].str.contains(r'\w+/\w+.*\.\w+', na=False)]

    if df.empty:
        df["is_trivial"] = False
        return df

    # --- Trivial patterns ---
    trivial_patterns = {
        "c":    [r'^(get|set|init|reset|free|alloc|load|save|open|close|input|output|error|message|complete)$', r'^(main)$'],
        "cpp":  [r'^(get|set|init|reset|copy|assign|release|load|save|open|close|input|output|error|message|complete)$', r'^(main|operator.*)$'],
        "cs":   [r'^(get|set|reset|dispose|clone|load|save|open|close|input|output|error|message|complete|is[A-Z][A-Za-z0-9_]*)$', r'^(Main)$'],
        "java": [r'^(get|set|load|save|open|close|input|output|error|message|complete|is[A-Z][A-Za-z0-9_]*|clone|toString|hashCode|equals)$', r'^(main)$'],
        "js":   [r'^(get|set|reset|constructor|load|save|open|close|input|output|error|message|complete)$', r'^(main)$'],
        "py":   [r'^__.*__$', r'^(init|get|set|reset|load|save|open|close|input|output|error|message|complete|is_[a-z0-9_]+)$', r'^(main)$'],
        "other":[r'^__.*__$', r'^(init|get|set|reset|load|save|open|close|input|output|error|message|complete|is_[a-z0-9_]+)$', r'^(main)$'],
    }

    compiled_patterns = {
        lang: re.compile("|".join(pats), re.IGNORECASE)
        for lang, pats in trivial_patterns.items()
    }

    # --- Flagging instead of filtering ---
    def is_trivial(name, lang):
        regex = compiled_patterns.get(lang)
        return bool(regex and regex.match(str(name)))

    df["is_trivial"] = df.apply(
        lambda row: is_trivial(row["method_name"], row["Language"]),
        axis=1
    )

    return df.reset_index(drop=True)


In [17]:
df1 = filter_trivial_functions_by_name(df)

In [18]:
df1.head()

,hash,project_id,version,license,method_name,file_location,repository_url,Language,is_trivial
0,ca823f7ade8864d7a6f911a3b1afeed7,344147451,1.756905e+12,Other,__iter__,./terratorch/samplers/single.py:21,https://github.com/IBM/terratorch/blob/e7eb482...,py,True
1,ea1f822ff9040aeaa5540719a0c30b64,344147451,1.756905e+12,Other,import_custom_modules,./terratorch/cli_tools.py:173,https://github.com/IBM/terratorch/blob/e7eb482...,py,False
2,eab1760c25814d020fc25e37144b3d16,344147451,1.756905e+12,Other,__iter__,./terratorch/samplers/single.py:34,https://github.com/IBM/terratorch/blob/e7eb482...,py,True
3,26adeb4519d2341006421b47d8786c9b,2129832116,1.756894e+12,Apache License 2.0,get_predicate_id,./src/unitxt/metrics.py:2890,https://github.com/IBM/unitxt/blob/fa5b604db4c...,py,False
4,a8aa31281dac29e85402060e8065ab48,2793013759,1.756850e+12,Apache License 2.0,write,./social-network/social-network-source/gen-cpp...,https://github.com/IBM/DeepRest/blob/9c368eeef...,cpp,False


In [19]:
df1.is_trivial.value_counts()

is_trivial
False    324603
True      27448
Name: count, dtype: int64

In [ ]:
#from global_trivial_methods import load_global_trivial_names
#trivial_names = load_global_trivial_names(file_path="../input_files/global_trivial_names.json", threshold=30)

In [23]:
from global_trivial_methods import load_global_trivial_names

def filter_with_global_trivial_names(df, file_path="global_trivial_names.json", threshold=20):
    """
    Update dataframe with trivial flag based on global trivial method names.
    If a row is not already trivial, mark it True if it matches global trivial names.
    """
    trivial_names = load_global_trivial_names(file_path, threshold)

    df = df.copy()

    # Ensure the column exists
    if "is_trivial" not in df.columns:
        df["is_trivial"] = False

    # Mark as trivial if method is in trivial_names and not already trivial
    df.loc[(~df["is_trivial"]) & (df["method_name"].isin(trivial_names)), "is_trivial"] = True

    return df.reset_index(drop=True)


In [24]:
df2 = filter_with_global_trivial_names(df1)

In [25]:
df2.is_trivial.value_counts()

is_trivial
False    324603
True      27448
Name: count, dtype: int64

In [ ]:
#df2.to_csv("../results/processed_data/trivial_method_count.csv", index=False)

In [27]:
df2.Language.value_counts()

Language
py      107725
c        97539
cpp      56583
cs       42338
java     30673
js       17193
Name: count, dtype: int64

In [17]:
df2 = pd.read_csv("../results/processed_data/trivial_method_count.csv")

In [18]:
# Cross-tab counts
counts = pd.crosstab(df2["Language"], df2["is_trivial"])

# Percentages per language
pct = counts.div(counts.sum(axis=1), axis=0) * 100

# Combine counts and percentages
summary = counts.astype(str) + " (" + pct.round(2).astype(str) + "%)"

# --- Add total column (False + True per row) ---
row_totals = counts.sum(axis=1)
summary["Total"] = row_totals.astype(str) + " (100%)"

# --- Add total row ---
total_counts = counts.sum()
total_pct = total_counts / total_counts.sum() * 100
total_row = total_counts.astype(str) + " (" + total_pct.round(2).astype(str) + "%)"
total_row["Total"] = str(row_totals.sum()) + " (100%)"
total_row.name = "Total"

summary = pd.concat([summary, pd.DataFrame([total_row])])

print(summary)

is_trivial           False            True          Total
c           96851 (99.29%)     688 (0.71%)   97539 (100%)
cpp         55179 (97.52%)    1404 (2.48%)   56583 (100%)
cs           39630 (93.6%)     2708 (6.4%)   42338 (100%)
java        27798 (90.63%)    2875 (9.37%)   30673 (100%)
js          16977 (98.74%)     216 (1.26%)   17193 (100%)
py          88168 (81.85%)  19557 (18.15%)  107725 (100%)
Total       324603 (92.2%)    27448 (7.8%)  352051 (100%)


In [19]:
df2.head()

,hash,project_id,version,license,method_name,file_location,repository_url,Language,is_trivial
0,ca823f7ade8864d7a6f911a3b1afeed7,344147451,1.756905e+12,Other,__iter__,./terratorch/samplers/single.py:21,https://github.com/IBM/terratorch/blob/e7eb482...,py,True
1,ea1f822ff9040aeaa5540719a0c30b64,344147451,1.756905e+12,Other,import_custom_modules,./terratorch/cli_tools.py:173,https://github.com/IBM/terratorch/blob/e7eb482...,py,False
2,eab1760c25814d020fc25e37144b3d16,344147451,1.756905e+12,Other,__iter__,./terratorch/samplers/single.py:34,https://github.com/IBM/terratorch/blob/e7eb482...,py,True
3,26adeb4519d2341006421b47d8786c9b,2129832116,1.756894e+12,Apache License 2.0,get_predicate_id,./src/unitxt/metrics.py:2890,https://github.com/IBM/unitxt/blob/fa5b604db4c...,py,False
4,a8aa31281dac29e85402060e8065ab48,2793013759,1.756850e+12,Apache License 2.0,write,./social-network/social-network-source/gen-cpp...,https://github.com/IBM/DeepRest/blob/9c368eeef...,cpp,False


In [20]:
df2=df2[df2["is_trivial"]==False]

In [22]:
df2.drop(columns=["is_trivial"], inplace=True)

In [23]:
df2.head()

,hash,project_id,version,license,method_name,file_location,repository_url,Language
1,ea1f822ff9040aeaa5540719a0c30b64,344147451,1.756905e+12,Other,import_custom_modules,./terratorch/cli_tools.py:173,https://github.com/IBM/terratorch/blob/e7eb482...,py
3,26adeb4519d2341006421b47d8786c9b,2129832116,1.756894e+12,Apache License 2.0,get_predicate_id,./src/unitxt/metrics.py:2890,https://github.com/IBM/unitxt/blob/fa5b604db4c...,py
4,a8aa31281dac29e85402060e8065ab48,2793013759,1.756850e+12,Apache License 2.0,write,./social-network/social-network-source/gen-cpp...,https://github.com/IBM/DeepRest/blob/9c368eeef...,cpp
5,a2e9f6c711a2bb0c33d4842c9c532384,2793013759,1.756850e+12,Apache License 2.0,send_UploadUrls,./social-network/social-network-source/gen-cpp...,https://github.com/IBM/DeepRest/blob/9c368eeef...,cpp
8,a6456cc040ea85dfaf675d69fc3679de,2793013759,1.756850e+12,Apache License 2.0,write,./social-network/social-network-source/gen-cpp...,https://github.com/IBM/DeepRest/blob/9c368eeef...,cpp


In [ ]:
"""
before = len(df2)
df2 = df2[(df2["project_id"] != -1) & (df2["version"] != -1)]
after = len(df2)

print(f"Dropped {before - after} rows. Remaining: {after}")
"""

Dropped 1008 rows. Remaining: 323595


In [29]:
df2.to_csv("../results/processed_data/trivial_method_filtered.csv", index=False)